In [1]:
# 1. 必要なライブラリのインストール
!pip install ripser persim umap-learn

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
import umap
from ripser import ripser
from persim import plot_diagrams, wasserstein
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# 2. 実験設定とディレクトリ作成
OUT_DIR = "H1_experiment_results"
os.makedirs(OUT_DIR, exist_ok=True)

# 試行回数（総生成要素数 N_samples * D_features）の6分割スケール
# 1,000(10^3) から 10,000,000(10^7) までの6段階
trial_scales = [1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

# TDAの計算量を現実的にするため、サンプル数(N)を固定し、次元数(D)を拡張して高次元空間の崩壊を再現する
N_SAMPLES = 400
results = []

print("=== H1計算実験を開始します ===")

# 3. メインループ（6分割スケール）
for i, total_trials in enumerate(trial_scales):
    step = i + 1
    # 総試行回数から次元数を逆算
    D_features = max(2, total_trials // N_SAMPLES)
    print(f"\n[Step {step}/6] 試行回数: {total_trials:,} (N={N_SAMPLES}, D={D_features})")

    # --- A. 初期値依存性を排除したベースライン生成 ---
    # U(0, 0.2) に従う一様分布（純粋なノイズ・同質集団のヌルモデル）
    np.random.seed(42 + step) # スケールごとにシードを変えるが、比較対照間では固定
    X_base = np.random.uniform(0, 0.2, size=(N_SAMPLES, D_features))

    # --- B. AHP的偏り・過分散の注入（特定の次元をバーストさせる） ---
    # 頻度上位5%の次元に対して、閾値を超える極端な重み（分散）を付与
    X_skewed = X_base.copy()
    hub_dims = max(1, int(D_features * 0.05))
    skew_multiplier = 10.0 + (step * 5.0) # スケールが上がるごとに偏りの圧力も強める
    X_skewed[:, :hub_dims] *= skew_multiplier

    # --- C. コサイン距離行列の計算 ---
    # ゼロ除算を避けるための微小値加算
    dist_base = pairwise_distances(X_base + 1e-10, metric='cosine')
    dist_skewed = pairwise_distances(X_skewed + 1e-10, metric='cosine')

    # --- D. UMAPによる次元圧縮（視覚的アーティファクトの確認） ---
    reducer = umap.UMAP(n_components=2, metric='precomputed', random_state=42)
    umap_base = reducer.fit_transform(dist_base)
    umap_skewed = reducer.fit_transform(dist_skewed)

    # --- E. パーシステントホモロジー（TDA）の計算 ---
    # H0（連結成分）と H1（穴）を計算
    ph_base = ripser(dist_base, distance_matrix=True, maxdim=1)['dgms']
    ph_skewed = ripser(dist_skewed, distance_matrix=True, maxdim=1)['dgms']

    # --- F. キャリブレーション（Wasserstein距離による位相的崩壊の検定） ---
    # ベースライン（同質）と偏り注入後（キメラ）のトポロジーの差分を定量化
    # H0, H1 が空配列の場合はダミーを入れる
    dgm_base_h0 = ph_base[0] if len(ph_base[0]) > 0 else np.array([[0, 0]])
    dgm_skewed_h0 = ph_skewed[0] if len(ph_skewed[0]) > 0 else np.array([[0, 0]])

    w_dist_h0 = wasserstein(dgm_base_h0, dgm_skewed_h0, matching=False)

    if len(ph_base)>1 and len(ph_skewed)>1 and len(ph_base[1])>0 and len(ph_skewed[1])>0:
        w_dist_h1 = wasserstein(ph_base[1], ph_skewed[1], matching=False)
    else:
        w_dist_h1 = 0.0

    # 結果の記録
    results.append({
        'Step': step,
        'Total_Trials': total_trials,
        'Dimensions': D_features,
        'Skew_Multiplier': skew_multiplier,
        'Wasserstein_H0': w_dist_h0,
        'Wasserstein_H1': w_dist_h1
    })

    # --- G. グラフ出力（都度保存） ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'Step {step}: Trials={total_trials:,} (D={D_features})', fontsize=16)

    # UMAP散布図
    axes[0,0].scatter(umap_base[:,0], umap_base[:,1], s=10, alpha=0.6)
    axes[0,0].set_title('UMAP: Baseline (Uniform)')
    axes[0,1].scatter(umap_skewed[:,0], umap_skewed[:,1], s=10, alpha=0.6, color='red')
    axes[0,1].set_title('UMAP: Skewed (Topology Collapsed)')

    # パーシステンス図
    plot_diagrams(ph_base, ax=axes[1,0])
    axes[1,0].set_title('PD: Baseline')
    plot_diagrams(ph_skewed, ax=axes[1,1])
    axes[1,1].set_title(f'PD: Skewed (W-Dist H0: {w_dist_h0:.3f})')

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/trial_step_{step}.png")
    plt.close()

# 4. 結果の集計とまとめグラフの作成
df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/summary_results.csv", index=False)

# Wasserstein距離の推移グラフ（キャリブレーション証明）
plt.figure(figsize=(8, 6))
plt.plot(df_results['Total_Trials'], df_results['Wasserstein_H0'], marker='o', label='Topology Deformation (H0 Wasserstein)')
plt.xscale('log')
plt.xlabel('Total Trials (N_samples * D_features, log scale)')
plt.ylabel('Wasserstein Distance vs Null Model')
plt.title('Topological Collapse Phase Transition')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/summary_collapse_plot.png")
plt.close()

# 5. ZIP圧縮とダウンロード
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(f"\n実験完了。全データを {OUT_DIR}.zip に圧縮しました。ダウンロードを開始します...")
files.download(f"{OUT_DIR}.zip")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.4 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18187 sha256=dbb2c6c54100b7a0a5b024ea632e28a5b237d5d977a826cefd0b823dae4b54b0
  Stored in directory: /root/.cache/pip/wheels/b3/32/77/3c78fadb9f917d5a5546ed0409574012401a83c544ec95999a
Successfully built hopcroftkarp
=== H1計算実験を開始します ===

[Step 1/6] 試行回数: 1,000 (N=400, D=2)

[Step 2/6] 試行回数: 10,000 (N=400, D=25)

[Step 3/6] 試行回数: 100,000 (N=400, D=250)

[Step 4/6] 試行回数: 1,000,000 (N=400, D=2500)

[Step 5/6] 試行回数: 5,000,000 (N=400, D=12500)

[Step 6/6] 試行回数: 10,000,000 (N=400, D=25000)

実験完了。全データを H1_experiment_results.zip に圧縮しました。ダウンロードを開始します...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>